In [1]:
import sys
import os
import numpy as np

# Thêm đường dẫn trỏ về python-rag-service để có thể import package 'app'
PROJECT_ROOT = os.path.abspath(os.path.join(os.path.dirname(__file__), "..", "python-rag-service"))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from langchain_huggingface import HuggingFaceEmbeddings
from app.settings import settings
from app.utils.reranker import Reranker
from app.utils.sparse_retriever import SparseRetriever

def calculate_chunk_scores(query: str, chunk_text: str):
    print("=" * 60)
    print(f"QUERY: {query}")
    print(f"CHUNK: {chunk_text}")
    print("=" * 60)
    
    # -------------------------------------------------------------
    # 1. DENSE SCORE (Cosine Similarity bằng Vector Embeddings)
    # -------------------------------------------------------------
    try:
        print("\n[1] Đang tính Dense Score...")
        embeddings = HuggingFaceEmbeddings(
            model_name=settings.embedding_model,
            model_kwargs={'trust_remote_code': True}
        )
        query_vector = embeddings.embed_query(query)
        chunk_vector = embeddings.embed_query(chunk_text)
        
        dot_product = np.dot(query_vector, chunk_vector)
        norm_q = np.linalg.norm(query_vector)
        norm_c = np.linalg.norm(chunk_vector)
        
        dense_score = (dot_product / (norm_q * norm_c)) if (norm_q * norm_c) != 0 else 0.0
        print(f"--> [Dense Score]: {dense_score:.4f}")
    except Exception as e:
        print(f"--> [Dense Score] Lỗi: {e}")

    # -------------------------------------------------------------
    # 2. SPARSE SCORE (BM25 với PyVi segmentation)
    # -------------------------------------------------------------
    try:
        print("\n[2] Đang tính Sparse Score (BM25)...")
        sparse_retriever = SparseRetriever()
        
        # Build index với duy nhất 1 chunk để tính BM25 local
        sparse_retriever.build_index([{"content": chunk_text}])
        results = sparse_retriever.retrieve_bm25(query, top_k=1)
        
        sparse_score = results[0]["score_bm25"] if results else 0.0
        print(f"--> [Sparse Score]: {sparse_score:.4f}")
    except Exception as e:
        print(f"--> [Sparse Score] Lỗi: {e}")

    # -------------------------------------------------------------
    # 3. RERANKER SCORE (Cross-Encoder / BGE-Reranker-M3)
    # -------------------------------------------------------------
    try:
        print("\n[3] Đang tính Reranker Score...")
        reranker = Reranker()
        candidates = [{"chunk_text": chunk_text}]
        
        results = reranker.rerank_candidates(query, candidates, top_n=1)
        reranker_score = results[0]["score_cross_encoder"] if results else 0.0
        print(f"--> [Reranker Score]: {reranker_score:.4f}")
    except Exception as e:
        print(f"--> [Reranker Score] Lỗi: {e}")
        
    print("\n" + "=" * 60)

if __name__ == "__main__":
    # Điền giá trị query và chunk_text test vào đây
    test_query = "Triệu chứng lâm sàng của bệnh ung thư bạch cầu cấp dòng Lympho là gì? "
    test_chunk = """LÂM SÀNG
    Biểu hiện không đặc hiệu, khởi phát bệnh một vài tuần đến một vài tháng.
    Mệt mỏi, chán ăn, sốt kéo dài, ra nhiều mồ hôi ban đêm, nhiễm trùng khó điều trị, thiếu
    máu, xuất huyết dưới da hoặc niêm mạc, gan, lách, hạch to, đau xương hoặc khớp. Biểu
    hiện hiếm gặp hơn: tăng áp lực nội sọ, liệt dây thần kinh sọ, khó thở do u trung thất, tinh
    hoàn to."""
    
    calculate_chunk_scores(query=test_query, chunk_text=test_chunk)


NameError: name '__file__' is not defined